# M1 — Loss & Class-Imbalance Study (Colab GPU runner)

Runs the loss sweep defined in `config/experiments/loss_study.yaml` on a Colab GPU and
writes ranked results into `docs/loss_study.md`.

**Before running:** `Runtime → Change runtime type → GPU (T4)`.

Losses compared: `bce` (pos_weight sweep), `dice`, `focal`, `dice_bce`. Model / data /
seed are held fixed — only the loss changes.


## 0. Confirm a GPU is attached


In [ ]:
!nvidia-smi

## 1. Clone the repository

Set `REPO_URL` / `BRANCH`. For a **private** repo, use a token URL:
`https://<GITHUB_TOKEN>@github.com/<owner>/<repo>.git`.


In [ ]:
REPO_URL = 'https://github.com/AliAgabalayev/Flood-Damage-Detection.git'  # <-- edit
BRANCH   = 'feature/m1-loss-imbalance-study'                               # <-- edit

import os
!git clone --branch $BRANCH --single-branch $REPO_URL flood
%cd flood
!git rev-parse --short HEAD

## 2. Install dependencies

Colab already ships a CUDA build of PyTorch, so we install only the extra packages and
**do not** touch `torch`/`torchvision` (avoids breaking CUDA).


In [ ]:
!pip -q install \
  'segmentation-models-pytorch==0.5.0' \
  'pytorch-lightning==2.6.5' \
  'torchmetrics==1.9.0' \
  'rasterio==1.5.0' \
  'albumentations==2.0.8' \
  'pydantic==2.13.4' \
  'mlflow==3.14.0' \
  'pyyaml==6.0.3'
print('deps installed')

## 3. Download the Sen1Floods11 hand-labeled data

Public GCS bucket — `gsutil` is preinstalled on Colab, no auth needed. ~700 MiB.


In [ ]:
!bash scripts/download_data.sh

### Sanity-check data & splits


In [ ]:
import pathlib
root = pathlib.Path('data/processed/sen1floods11')
print('S1Hand   :', len(list((root/'S1Hand').glob('*.tif'))), 'tif')
print('LabelHand:', len(list((root/'LabelHand').glob('*.tif'))), 'tif')
for s in ('train','val','test'):
    p = pathlib.Path('data/splits/official')/f'{s}.csv'
    print(f'{s:5}:', sum(1 for _ in open(p)) if p.exists() else 'MISSING', 'rows')

## 4. Run the loss sweep

~7 runs. Config `device: cuda` is already set. Re-running skips runs already present in
`experiments/loss_study/results.csv`, so you can resume if the session drops.

_Tip:_ shorten with `--only dice focal` while testing.


In [ ]:
!PYTHONPATH=src python scripts/run_loss_study.py

## 5. Build the ranked results table and update the report


In [ ]:
!PYTHONPATH=src python scripts/make_results_table.py --update-report
print('
--- docs/loss_study.md (results section) ---')
import re
txt = open('docs/loss_study.md').read()
print(txt.split('<!-- RESULTS:START -->')[1].split('<!-- RESULTS:END -->')[0])

## 6. Download the results

Grab the raw CSV (and optionally commit the updated report from your machine).


In [ ]:
from google.colab import files
files.download('experiments/loss_study/results.csv')

---
### Optional — commit the updated report back to the branch
```bash
!git config user.email you@example.com && git config user.name you
!git add docs/loss_study.md && git commit -m 'M1: loss study results' && git push
```
